In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalJupyterSpark")
    .master("local[*]")  # ✅ 本地模式，不连 K8s cluster
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # ✅ 现有的 catalog: local
    # .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    # .config("spark.sql.catalog.local.type", "hadoop")
    # .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/")
    # ✅ 新增的 catalog: standardized
    .config("spark.sql.catalog.standardized", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.standardized.type", "hadoop")
    .config("spark.sql.catalog.standardized.warehouse", "s3a://bc2-raw-restricted-ide/")
    
    # ✅ 通过 port-forward 访问 K8s 里的 MinIO
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    # ✅ 通过 Minikube IP + NodePort 访问 K8s 里的 MinIO
    #.config("spark.hadoop.fs.s3a.endpoint", "http://192.168.49.2:30900")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
spark

your 131072x1 screen size is bogus. expect trouble
26/05/10 20:24:06 WARN Utils: Your hostname, DESKTOP-CDCLH86 resolves to a loopback address: 127.0.1.1; using 172.22.19.65 instead (on interface eth0)
26/05/10 20:24:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/phil/ldp/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/phil/.ivy2/cache
The jars for the packages stored in: /home/phil/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bb3d84e8-47e7-4c5f-8370-a2f5220c6dcb;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 158ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.

Spark Version: 3.5.3
Spark UI: http://172.22.19.65:4040


In [6]:

df = spark.sql("SHOW CATALOGS")
df.show()
print("正常会显示spark session 中创建的catalog：standardized")

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

正常会显示spark session 中创建的catalog：standardized


In [7]:
spark.sql("USE standardized")

26/05/10 20:24:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [8]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS standardized.bc2_cde_rstk")

DataFrame[]

In [1]:
#spark.sql("SHOW NAMESPACES IN standardized").show()

In [9]:
spark.sql("DROP NAMESPACE IF EXISTS standardized.bc2_cde_rstk CASCADE")


DataFrame[]

In [11]:
print(spark.conf.get("spark.sql.catalog.standardized", "NOT FOUND"))


org.apache.iceberg.spark.SparkCatalog


In [12]:
# 1. 确保 Namespace 存在
spark.sql("CREATE NAMESPACE IF NOT EXISTS standardized.bc2_cde_rstk")

# 2. 执行建表 DDL (加上 standardized. 前缀)
spark.sql("""
CREATE TABLE IF NOT EXISTS standardized.bc2_cde_rstk.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER(
    RTN_RESP_EV_ID                          STRING,
    AI_ENT_CD                               STRING,
    RPT_POS_DT                              DATE,
    RTN_ID                                  STRING,
    SBMN_DTTM                               TIMESTAMP,
    HALF_YR_END_DT                          DATE,
    APPR_MONY_BRKR_NM                       STRING,
    HK_RELVNT_BUSN_REVN_AMT                 DECIMAL(30,10),
    OTH_REVN_SFC_LIC_RA_AMT                 DECIMAL(30,10),
    OTH_REVN_SFC_SUTHORIZED_ATS_AMT         DECIMAL(30,10),
    OTH_REVN_NON_MM_OTC_DRTV_DEAL_AMT	    DECIMAL(30,10),
    OTH_REVN_ADV_OTC_DRTV_AMT	            DECIMAL(30,10),
    OTH_REVN_OTH_INCM_AMT                   DECIMAL(30,10),
    OTH_REVN_OTH_INCM_MAJ_CAT_1_TXT     	STRING,
    OTH_REVN_OTH_INCM_MAJ_CAT_2_TXT     	STRING,
    OTH_REVN_OTH_INCM_MAJ_CAT_3_TXT	        STRING,
    OTH_REVN_OTH_INCM_MAJ_CAT_4_TXT	        STRING,
    TOT_REVN_AMT                            DECIMAL(30,10),
    OPRT_EXP_STAF_EXP_AMT                   DECIMAL(30,10),
    OPRT_EXP_RNTAL_EXP_AMT                  DECIMAL(30,10),
    OPRT_EXP_ENTRTMT_EXP_AMT                DECIMAL(30,10),
    OPRT_EXP_OTH_OPRT_EXP_AMT               DECIMAL(30,10),
    TOT_OPRT_EXP_AMT                        DECIMAL(30,10),
    OPRT_PRFT_AMT                           DECIMAL(30,10),
    EXCPTN_ITEM_AMT                         DECIMAL(30,10),
    BEF_INT_PRFT_AMT                        DECIMAL(30,10),
    NET_INT_PYBL_AMT                        DECIMAL(30,10),
    BEF_TAX_PRFT_AMT                        DECIMAL(30,10),
    LESS_TAX_PRVSN_NET_CHRG_AMT             DECIMAL(30,10),
    AFTER_TAX_PRFT_AMT                      DECIMAL(30,10),
    XTRY_ITEM_AMT                           DECIMAL(30,10),
    LESS_OTHER_RESV_TFR_AMT                 DECIMAL(30,10),
    PER_PRFT_AMT                            DECIMAL(30,10),
    DCLR_DVD_AMT                            DECIMAL(30,10),
    ROW_VLD_STS_CD                          STRING,
    ROW_VLD_MSG_TXT                         STRING,
    TX_DT                                   DATE,
    INSE_DTTM                               TIMESTAMP,
    UPDT_DTTM                               TIMESTAMP
) USING iceberg 
TBLPROPERTIES (
    'write.parquet.compression-codec' = 'snappy',
    'format-version' = '2'
)
""")

print("表： AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER 已在 standardized 目录中成功建立！")

26/05/10 20:26:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


表： AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER 已在 standardized 目录中成功建立！


In [13]:
#006.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER
# 执行建表 DDL
spark.sql("""
CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER(
    RTN_RESP_EV_ID                          VARCHAR(100),
    AI_ENT_CD                               VARCHAR(100),
    RPT_POS_DT                              DATE,
    RTN_ID                                  VARCHAR(50),
    SBMN_DTTM                               TIMESTAMP,
    HALF_YR_END_DT                          DATE,
    APPR_MONY_BRKR_NM                       VARCHAR(2000),
    HK_RELVNT_BUSN_REVN_AMT                 DECIMAL(30,10),
    OTH_REVN_SFC_LIC_RA_AMT                 DECIMAL(30,10),
    OTH_REVN_SFC_SUTHORIZED_ATS_AMT         DECIMAL(30,10),
    OTH_REVN_NON_MM_OTC_DRTV_DEAL_AMT	    DECIMAL(30,10),
    OTH_REVN_ADV_OTC_DRTV_AMT	            DECIMAL(30,10),
    OTH_REVN_OTH_INCM_AMT                   DECIMAL(30,10),
    OTH_REVN_OTH_INCM_MAJ_CAT_1_TXT     	VARCHAR(2000),
    OTH_REVN_OTH_INCM_MAJ_CAT_2_TXT     	VARCHAR(2000),
    OTH_REVN_OTH_INCM_MAJ_CAT_3_TXT	        VARCHAR(2000),
    OTH_REVN_OTH_INCM_MAJ_CAT_4_TXT	        VARCHAR(2000),
    TOT_REVN_AMT                            DECIMAL(30,10),
    OPRT_EXP_STAF_EXP_AMT                   DECIMAL(30,10),
    OPRT_EXP_RNTAL_EXP_AMT                  DECIMAL(30,10),
    OPRT_EXP_ENTRTMT_EXP_AMT                DECIMAL(30,10),
    OPRT_EXP_OTH_OPRT_EXP_AMT               DECIMAL(30,10),
    TOT_OPRT_EXP_AMT                        DECIMAL(30,10),
    OPRT_PRFT_AMT                           DECIMAL(30,10),
    EXCPTN_ITEM_AMT                         DECIMAL(30,10),
    BEF_INT_PRFT_AMT                        DECIMAL(30,10),
    NET_INT_PYBL_AMT                        DECIMAL(30,10),
    BEF_TAX_PRFT_AMT                        DECIMAL(30,10),
    LESS_TAX_PRVSN_NET_CHRG_AMT             DECIMAL(30,10),
    AFTER_TAX_PRFT_AMT                      DECIMAL(30,10),
    XTRY_ITEM_AMT                           DECIMAL(30,10),
    LESS_OTHER_RESV_TFR_AMT                 DECIMAL(30,10),
    PER_PRFT_AMT                            DECIMAL(30,10),
    DCLR_DVD_AMT                            DECIMAL(30,10),
    ROW_VLD_STS_CD                          VARCHAR(255),
    ROW_VLD_MSG_TXT                         VARCHAR(2000),
    TX_DT                                   DATE,
    INSE_DTTM                               TIMESTAMP,
    UPDT_DTTM                               TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER 已成功建立！")

表： AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER 已成功建立！


In [14]:
#007.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF
# 执行建表 DDL
spark.sql("""
CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF(
    RTN_RESP_EV_ID                          VARCHAR(100), 
    AI_ENT_CD                               VARCHAR(100), 
    RPT_POS_DT                              DATE, 
    RTN_ID                                  VARCHAR(50),
    SBMN_DTTM                               TIMESTAMP, 
    HALF_YR_END_DT                          DATE          ,
    APPR_MONY_BRKR_NM                       VARCHAR(2000) , 
    TNGBL_ASSET_AMT                         DECIMAL(30,10), 
    INV_AMT                                 DECIMAL(30,10), 
    OTH_NON_CUR_ASSET_NM                    VARCHAR(2000) , 
    OTH_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    TOT_NON_CUR_ASSET_AMT                   DECIMAL(30,10), 
    CASH_AT_BANK_AND_IN_HAND_AMT            DECIMAL(30,10), 
    DEBTOR_AMT                              DECIMAL(30,10), 
    OTH_CUR_ASSET_NM                        VARCHAR(2000) , 
    OTH_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_CUR_ASSET_AMT                       DECIMAL(30,10), 
    TOT_ASSET_AMT                           DECIMAL(30,10), 
    CROR_AMT                                DECIMAL(30,10), 
    LN_AMT                                  DECIMAL(30,10), 
    DFR_TAX_AMT                             DECIMAL(30,10), 
    OTH_CUR_LIAB_NM                         VARCHAR(2000) , 
    OTH_CUR_LIAB_AMT                        DECIMAL(30,10), 
    TOT_CUR_LIAB_AMT                        DECIMAL(30,10), 
    LONG_TERM_LN_AMT                        DECIMAL(30,10), 
    OTH_LONG_TERM_LIAB_NM                   VARCHAR(2000) , 
    OTH_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LONG_TERM_LIAB_AMT                  DECIMAL(30,10), 
    TOT_LIAB_AMT                            DECIMAL(30,10), 
    NET_ASSET_AMT                           DECIMAL(30,10), 
    PD_UP_SHR_CAP_AMT                       DECIMAL(30,10), 
    SHR_PREM_ACCT_AMT                       DECIMAL(30,10),
    WOUT_SHR_ISSNC_SHR_HLD_CAP_CNTRB_AMT	decimal(30,10),
    REVALQ_RESV_AMT                         DECIMAL(30,10), 
    OTH_RESV_NM                             VARCHAR(2000) , 
    OTH_RESV_AMT                            DECIMAL(30,10), 
    PRFT_AND_LOSS_ACCT_AMT                  DECIMAL(30,10), 
    TOT_SHRHLD_FUND_AMT                     DECIMAL(30,10), 
    ROW_VLD_STS_CD                          VARCHAR(255)  , 
    ROW_VLD_MSG_TXT                         VARCHAR(2000) ,
    TX_DT                                   DATE, 
    INSE_DTTM                               TIMESTAMP,       
    UPDT_DTTM                               TIMESTAMP       
) USING iceberg 
TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！")

表： AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF 已成功建立！


In [15]:
#008.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN
# 执行建表 DDL
spark.sql("""
CREATE TABLE IF NOT EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN(
    RTN_RESP_EV_ID                                                      VARCHAR(100),
    AI_ENT_CD                                                           VARCHAR(100),
    RPT_POS_DT                                                          DATE,
    RTN_ID                                                              VARCHAR(50),
    SBMN_DTTM                                                           TIMESTAMP,
    HALF_YR_END_DT                                                      DATE,
    APPR_MONY_BRKR_NM                                                   VARCHAR(2000),
    TRAN_BSS_REVN_GEN_IND                                               VARCHAR(100),
    HK_RELVNT_BUSN_REVN_AMT                                             DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_DEP_AND_LN_AMT                                 DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_FX_SPOT_FRWD_FUT_AMT                           DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_LVR_FX_CNTC_AMT                                DECIMAL(30,10),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_NM                          VARCHAR(2000),
    TRDNL_MONY_MKT_INSTM_OTH_MONY_MKT_INSTM_AMT                         DECIMAL(30,10),
    FX_DRTV_FX_SWAP_AMT                                                 DECIMAL(30,10),
    FX_DRTV_NDF_AMT                                                     DECIMAL(30,10),
    FX_DRTV_NDO_AMT                                                     DECIMAL(30,10),
    FX_DRTV_VANILLA_FX_OPT_AMT                                          DECIMAL(30,10),
    FX_DRTV_OTH_FX_DRTV_NM                                              VARCHAR(2000),
    FX_DRTV_OTH_FX_DRTV_AMT                                             DECIMAL(30,10),
    INT_RT_DRTV_SNGL_CURY_IRS_AMT                                       DECIMAL(30,10),
    INT_RT_DRTV_CROSS_CURY_IRS_AMT                                      DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_CAP_FLOOR_AMT                                    DECIMAL(30,10),
    INT_RT_DRTV_FRA_AMT                                                 DECIMAL(30,10),
    INT_RT_DRTV_INT_RT_OPT_AMT                                          DECIMAL(30,10),
    INT_RT_DRTV_OTH_INT_RT_DRTV_NM                                      VARCHAR(2000),
    INT_RT_DRTV_OTH_INT_RT_DRTV_AMT                                     DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_I_NM                                          VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_I_AMT                                         DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_II_NM                                         VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_II_AMT                                        DECIMAL(30,10),
    OTH_MONY_MKT_OTC_DRTV_III_NM                                        VARCHAR(2000),
    OTH_MONY_MKT_OTC_DRTV_III_AMT                                       DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_DEAL_IN_SCTY_AMT                           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_CLNT_CLRG_SRVC_FOR_OTC_DRTV_TRAN           DECIMAL(30,10),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_NM                                VARCHAR(2000),
    BUSN_REL_ON_EXPT_UND_SFO_OTH_BUSN_AMT                               DECIMAL(30,10),
    TOT_HK_RELVNT_BUSN_TRAN_AMT                                         DECIMAL(30,10),
    ROW_VLD_STS_CD                                                      VARCHAR(255),
    ROW_VLD_MSG_TXT                                                     VARCHAR(2000),
    TX_DT                                                               DATE,
    INSE_DTTM                                                           TIMESTAMP,
    UPDT_DTTM                                                           TIMESTAMP
) USING iceberg TBLPROPERTIES (
    "write.parquet.compression-codec" = "snappy",
    "write.spark.accept-any-schema" = "true"
);

""")

print("表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！")

表： AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN 已成功建立！


In [ ]:
#DROP TABLE [IF EXISTS] table_name;

# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
# spark.sql("DROP TABLE IF EXISTS bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")

# spark.sql("TRUNCATE TABLE standardized.bc2_cde_rstk.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER;")
# spark.sql("TRUNCATE TABLE standardized.bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF;")
# spark.sql("TRUNCATE TABLE standardized.bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN;")



In [16]:
#Check Table status before start ETL

print("AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER""").show()
print("AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IIB_FIN_POS_AS_AT_THE_END_OF_THE_HALF""").show()
print("AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN")
spark.sql("""select * from bc2_cde_rstk.AMB_PART_IV_REVN_FROM_HK_RELVNT_BUSN""").show()




AMB_PART_IIA_PRFT_AND_LOSS_FOR_THE_HALF_YR_PER
+--------------+---------+----------+------+---------+--------------+-----------------+-----------------------+-----------------------+-------------------------------+---------------------------------+-------------------------+---------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+------------+---------------------+----------------------+------------------------+-------------------------+----------------+-------------+---------------+----------------+----------------+----------------+---------------------------+------------------+-------------+-----------------------+------------+------------+--------------+---------------+-----+---------+---------+
|RTN_RESP_EV_ID|AI_ENT_CD|RPT_POS_DT|RTN_ID|SBMN_DTTM|HALF_YR_END_DT|APPR_MONY_BRKR_NM|HK_RELVNT_BUSN_REVN_AMT|OTH_REVN_SFC_LIC_RA_AMT|OTH_REVN_SFC_SUTHORIZED_ATS_AMT|OTH_REVN_NON_MM_OTC_DRTV_DEAL_AMT

In [ ]:
#Check parquets before start ETL

print("data of iia")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iia-prft-and-loss-for-the-half-yr-per_1_CHC101_20260430_AMB_20260430000001_Passed_20260430.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iia-prft-and-loss-for-the-half-yr-per_1_CHC101_20260430_AMB_20260501000001_Passed_20260501.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iia-prft-and-loss-for-the-half-yr-per_1_CHC101_20260502_AMB_20260502000001_Passed_20260502.parquet`;""").show()

print("data of iib")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20260430_AMB_20260430000001_Passed_20260430.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20260430_AMB_20260501000001_Passed_20260501.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iib-fin-pos-as-at-the-end-of-the-half_1_CHC101_20260502_AMB_20260502000001_Passed_20260502.parquet`;""").show()

print("data of iv")
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20260430_AMB_20260430000001_Passed_20260430.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20260430_AMB_20260501000001_Passed_20260501.parquet`;""").show()
spark.sql("""SELECT * FROM parquet.`s3a://bc2-raw-restricted-ide/20260507/bc2_amb-part-iv-revn-from-hk-relvnt-busn_1_CHC101_20260502_AMB_20260502000001_Passed_20260502.parquet`;""").show()

